# Tabela de dicionários

In [0]:
# Criando um dataframe spark sem inferência de tipos, para o conjunto de dados fuzzy_factory_data_dictionary
df_dicionario = spark.read.csv(
    "/Volumes/workspace/datasets/dicionario/fuzzy_factory_data_dictionary.csv",  
    header=True,
    inferSchema=False
)
# Salvando o dataframe como tabela para visualização dos detalhes
df_dicionario.write.mode("overwrite").saveAsTable("dicionario")


In [0]:
%sql
-- Exibindos os valores contidos na tabela para consulta das descrições
SELECT * FROM dicionario

table,field,description
orders,order_id,Unique identifier for each order (PK)
orders,created_at,Timestamp when the order was placed
orders,website_session_id,Unique identifier for the website session (FK)
orders,user_id,Unique identifier for the user (FK)
orders,primary_product_id,Unique identifier for the primary product in the order if part of a bundle (FK)
orders,items_purchased,Number of items in the order
orders,price_usd,Total price for the items in the order
orders,cogs_usd,Cost of goods sold for the items in the order
order_items,order_item_id,Unique identifier for each order item (PK)
order_items,created_at,Timestamp when the order was placed


In [0]:
%sql
-- Excluindo a tabela já que ela não tem mais utilidade
DROP TABLE dicionario

# Criação de tabelas

In [0]:
lista_conjuntos = ['order_item_refunds', 'order_items', 'orders', 'products', 'website_pageviews', 'website_sessions'] # Listando os nomes de cada conjunto.
# Iteração para cada nome contido na lista nomes
for conjunto in lista_conjuntos:
    # Leitura do conjunto csv referente ao nome contido na lista e criação de um Dataframe spark sem inferência de tipos.
    df = spark.read.csv(
        f"/Volumes/workspace/datasets/{conjunto}/{conjunto}.csv",  
        header=True,
        inferSchema=False
    )
    # Tranformando o dataframe em uma tabela no schema workspace.data_tables_s. Caso ja exista uma tabela com o mesmo nome, é gerado erro de duplicidade então é usado o modo overwrite para o tratamento da excessão.
    try:
        df.write.saveAsTable(f"workspace.data_tables_s.{conjunto}")
    except:
        df.write.mode("overwrite").saveAsTable(f"workspace.data_tables_s.{conjunto}")

In [0]:
%sql
-- Exibindo as tabelas criadas
SHOW TABLES IN data_tables_s

database,tableName,isTemporary
data_tables_s,order_item_refunds,false
data_tables_s,order_items,false
data_tables_s,orders,false
data_tables_s,products,false
data_tables_s,website_pageviews,false
data_tables_s,website_sessions,false


## order_item_refunds

In [0]:
%sql

SELECT * FROM data_tables_s.order_item_refunds

order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd
1,2012-04-06 11:32:43,57,57,49.99
2,2012-04-13 01:09:43,74,74,49.99
3,2012-04-15 07:03:48,71,71,49.99
4,2012-04-17 20:00:37,118,118,49.99
5,2012-04-22 20:53:49,116,116,49.99
6,2012-05-04 11:59:07,147,147,49.99
7,2012-05-12 02:41:14,186,186,49.99
8,2012-05-16 13:06:01,191,191,49.99
9,2012-05-24 16:00:09,179,179,49.99
10,2012-05-30 17:20:44,199,199,49.99


In [0]:
%sql
-- selecionando todos os dados que possam conter valores NULL para tratá-los.
SELECT * FROM data_tables_s.order_item_refunds WHERE created_at IS NULL OR order_item_id IS NULL OR order_id IS NULL OR refund_amount_usd IS NULL

order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd


In [0]:
%sql
-- Realizando a inserção de valores padrões nos campos vazios que não sejam ids
UPDATE data_tables_s.order_item_refunds SET created_at = '1900-01-01 00:00:00' WHERE created_at IS NULL;
UPDATE data_tables_s.order_item_refunds SET refund_amount_usd = '0.00' WHERE refund_amount_usd IS NULL;

num_affected_rows
0


In [0]:
%sql
-- Criando a tabela com os dados da tabela order_item_refunds com os tipos de dados corretos
CREATE TABLE IF NOT EXISTS data_tables_s.order_item_refunds_typed AS
SELECT
  CAST(order_item_refund_id AS INT) AS order_item_refund_id,
  DATE_FORMAT(CAST(created_at AS TIMESTAMP), 'yyyy-MM-dd HH:mm:ss') AS created_at,
  CAST(order_id AS INT) AS order_id,
  CAST(order_item_id AS INT) AS order_item_id,
  CAST(refund_amount_usd AS DECIMAL(6, 2)) AS refund_amount_usd
FROM data_tables_s.order_item_refunds;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Exibindo os valores minimos e maximos de todas as colunas menos a order_item_refund_id
SELECT MIN(created_at) AS Minimo_created_at,
       MAX(created_at) AS Maximo_created_at,
       MIN(order_item_id) AS Minimo_order_item_id,
       MAX(order_item_id) AS Maximo_order_item_id,
       MIN(order_id) AS Minimo_order_id,
       MAX(order_id) AS Maximo_order_id,
       MIN(refund_amount_usd) AS Minimo_refund_amount_usd,
       MAX(refund_amount_usd) AS Maximo_refund_amount_usd
 FROM data_tables_s.order_item_refunds_typed;

Minimo_created_at,Maximo_created_at,Minimo_order_item_id,Maximo_order_item_id,Minimo_order_id,Maximo_order_id,Minimo_refund_amount_usd,Maximo_refund_amount_usd
2012-04-06 11:32:43,2015-04-01 18:11:08,57,39950,57,32255,29.99,59.99


## order_items

In [0]:
%sql
-- Exibindo os dados da tabela
SELECT * FROM data_tables_s.order_items

order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
1,2012-03-19 10:42:46,1,1,1,49.99,19.49
2,2012-03-19 19:27:37,2,1,1,49.99,19.49
3,2012-03-20 06:44:45,3,1,1,49.99,19.49
4,2012-03-20 09:41:45,4,1,1,49.99,19.49
5,2012-03-20 11:28:15,5,1,1,49.99,19.49
6,2012-03-20 16:12:47,6,1,1,49.99,19.49
7,2012-03-20 17:03:41,7,1,1,49.99,19.49
8,2012-03-20 23:35:27,8,1,1,49.99,19.49
9,2012-03-21 02:35:01,9,1,1,49.99,19.49
10,2012-03-21 06:45:58,10,1,1,49.99,19.49


In [0]:
%sql
-- Selecionando todos os dados que possam conter valores NULL para tratá-los.
SELECT * FROM data_tables_s.order_items WHERE created_at IS NULL OR order_id IS NULL OR product_id IS NULL OR is_primary_item IS NULL OR price_usd IS NULL OR cogs_usd IS NULL

order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd


In [0]:
%sql
-- Realizando a inserção de valores padrões nos campos vazios que não sejam ids
UPDATE data_tables_s.order_items SET created_at = '1900-01-01 00:00:00' WHERE created_at IS NULL;
UPDATE data_tables_s.order_items SET is_primary_item = '-1' WHERE is_primary_item IS NULL;
UPDATE data_tables_s.order_items SET price_usd = '0.00' WHERE price_usd IS NULL;
UPDATE data_tables_s.order_items SET cogs_usd = '0.00' WHERE cogs_usd IS NULL;


num_affected_rows
0


In [0]:
%sql
-- Criando a tabela com os dados da tabela order_item com os tipos de dados corretos
CREATE TABLE IF NOT EXISTS data_tables_s.order_items_typed AS
SELECT
  CAST(order_item_id AS INT) AS order_item_id,
  DATE_FORMAT(CAST(created_at AS TIMESTAMP), 'yyyy-MM-dd HH:mm:ss') AS created_at,
  CAST(order_id AS INT) AS order_id,
  CAST(product_id AS INT) AS product_id,
  CAST(is_primary_item AS INT) AS is_primary_item,
  CAST(price_usd AS DECIMAL(6, 2)) AS price_usd,
  CAST(cogs_usd AS DECIMAL(6, 2)) AS cogs_usd
FROM data_tables_s.order_items;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Exibindo os valores minimos e maximos de todas as colunas menos a order_item_id
SELECT MIN(created_at) AS Minimo_created_at,
       MAX(created_at) AS Maximo_created_at,
       MIN(product_id) AS Minimo_product_id,
       MAX(product_id) AS Maximo_product_id,
       MIN(order_id) AS Minimo_order_id,
       MAX(order_id) AS Maximo_order_id,
       MIN(is_primary_item) AS Minimo_primary_item,
       MAX(is_primary_item) AS Maximo_primary_item,
       MIN(price_usd) AS Minimo_price_usd,
       MAX(price_usd) AS Maximo_price_usd,
       MIN(cogs_usd) AS Minimo_cogs_usd,
       MAX(cogs_usd) AS Maximo_cogs_usd
 FROM data_tables_s.order_items_typed;

Minimo_created_at,Maximo_created_at,Minimo_product_id,Maximo_product_id,Minimo_order_id,Maximo_order_id,Minimo_primary_item,Maximo_primary_item,Minimo_price_usd,Maximo_price_usd,Minimo_cogs_usd,Maximo_cogs_usd
2012-03-19 10:42:46,2015-03-19 05:38:31,1,4,1,32313,0,1,29.99,59.99,9.49,22.49


## orders

In [0]:
%sql
-- Exibindo os dados da tabela
SELECT * FROM data_tables_s.orders LIMIT 30

order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49
2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49
3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49
4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49
5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49
6,2012-03-20 16:12:47,232,232,1,1,49.99,19.49
7,2012-03-20 17:03:41,241,241,1,1,49.99,19.49
8,2012-03-20 23:35:27,295,295,1,1,49.99,19.49
9,2012-03-21 02:35:01,304,304,1,1,49.99,19.49
10,2012-03-21 06:45:58,317,317,1,1,49.99,19.49


In [0]:
%sql
-- Selecionando todos os dados que possam conter valores NULL para tratá-los.
SELECT * FROM data_tables_s.orders WHERE created_at IS NULL OR website_session_id IS NULL OR user_id IS NULL OR primary_product_id IS NULL OR items_purchased IS NULL OR price_usd IS NULL OR cogs_usd IS NULL

order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd


In [0]:
%sql
-- Realizando a inserção de valores padrões nos campos vazios que não sejam ids
UPDATE data_tables_s.orders SET created_at = '1900-01-01 00:00:00' WHERE created_at IS NULL;
UPDATE data_tables_s.orders SET items_purchased = '0' WHERE items_purchased IS NULL;
UPDATE data_tables_s.orders SET price_usd = '0.00' WHERE price_usd IS NULL;
UPDATE data_tables_s.orders SET cogs_usd = '0.00' WHERE cogs_usd IS NULL;


num_affected_rows
0


In [0]:
%sql
-- Criando a tabela com os dados da tabela order_item com os tipos de dados corretos
CREATE TABLE IF NOT EXISTS data_tables_s.orders_typed AS
SELECT
  CAST(order_id AS INT) AS order_id,
  DATE_FORMAT(CAST(created_at AS TIMESTAMP), 'yyyy-MM-dd HH:mm:ss') AS created_at,
  CAST(website_session_id AS INT) AS website_session_id,
  CAST(user_id AS INT) AS user_id,
  CAST(primary_product_id AS INT) AS primary_product_id,
  CAST(items_purchased AS INT) AS items_purchased,
  CAST(price_usd AS DECIMAL(6, 2)) AS price_usd,
  CAST(cogs_usd AS DECIMAL(6, 2)) AS cogs_usd
FROM data_tables_s.orders;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Exibindo os valores minimos e maximos de todas as colunas menos a order_item_id
SELECT MIN(created_at) AS Minimo_created_at,
       MAX(created_at) AS Maximo_created_at,
       MIN(website_session_id) AS Minimo_website_session_id,
       MAX(website_session_id) AS Maximo_website_session_id,
       MIN(user_id) AS Minimo_user_id,
       MAX(user_id) AS Maximo_user_id,
       MIN(primary_product_id) AS Minimo_product_id,
       MAX(primary_product_id) AS Maximo_product_id,
       MIN(items_purchased) AS Minimo_items_purchased,
       MAX(items_purchased) AS Maximo_items_purchased,
       MIN(price_usd) AS Minimo_price_usd,
       MAX(price_usd) AS Maximo_price_usd,
       MIN(cogs_usd) AS Minimo_cogs_usd,
       MAX(cogs_usd) AS Maximo_cogs_usd
 FROM data_tables_s.orders_typed;

Minimo_created_at,Maximo_created_at,Minimo_website_session_id,Maximo_website_session_id,Minimo_user_id,Maximo_user_id,Minimo_product_id,Maximo_product_id,Minimo_items_purchased,Maximo_items_purchased,Minimo_price_usd,Maximo_price_usd,Minimo_cogs_usd,Maximo_cogs_usd
2012-03-19 10:42:46,2015-03-19 05:38:31,20,472818,13,394273,1,4,1,2,29.99,109.98,9.49,41.98


## Products

In [0]:
%sql
-- Exibindo os dados da tabela
SELECT * FROM data_tables_s.products

product_id,created_at,product_name
1,2012-03-19 08:00:00,The Original Mr. Fuzzy
2,2013-01-06 13:00:00,The Forever Love Bear
3,2013-12-12 09:00:00,The Birthday Sugar Panda
4,2014-02-05 10:00:00,The Hudson River Mini bear


In [0]:
%sql
-- Criando a tabela com os dados da tabela order_item com os tipos de dados corretos
CREATE TABLE IF NOT EXISTS data_tables_s.products_typed AS
SELECT
  CAST(product_id AS INT) AS product_id,
  DATE_FORMAT(CAST(created_at AS TIMESTAMP), 'yyyy-MM-dd HH:mm:ss') AS created_at,
  CAST(product_name AS STRING) AS product_name
FROM data_tables_s.products;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Exibindo os valores da nova tabela
SELECT * FROM data_tables_s.products_typed;

product_id,created_at,product_name
1,2012-03-19 08:00:00,The Original Mr. Fuzzy
2,2013-01-06 13:00:00,The Forever Love Bear
3,2013-12-12 09:00:00,The Birthday Sugar Panda
4,2014-02-05 10:00:00,The Hudson River Mini bear


## website_pageviews

In [0]:
%sql
-- Exibindo os dados da tabela com limite de 30 registros
SELECT * FROM data_tables_s.website_pageviews LIMIT 30

website_pageview_id,created_at,website_session_id,pageview_url
1,2012-03-19 08:04:16,1,/home
2,2012-03-19 08:16:49,2,/home
3,2012-03-19 08:26:55,3,/home
4,2012-03-19 08:37:33,4,/home
5,2012-03-19 09:00:55,5,/home
6,2012-03-19 09:05:46,6,/home
7,2012-03-19 09:06:27,7,/home
8,2012-03-19 09:10:08,6,/products
9,2012-03-19 09:10:52,6,/the-original-mr-fuzzy
10,2012-03-19 09:14:02,6,/cart


In [0]:
%sql
-- Selecionando todos os dados que possam conter valores NULL para tratá-los.
SELECT * FROM data_tables_s.website_pageviews WHERE created_at IS NULL OR website_session_id IS NULL OR pageview_url IS NULL

website_pageview_id,created_at,website_session_id,pageview_url


In [0]:
%sql
-- Criando a tabela com os dados da tabela order_item com os tipos de dados corretos
CREATE TABLE IF NOT EXISTS data_tables_s.website_pageviews_typed AS
SELECT
  CAST(website_pageview_id AS INT) AS website_pageview_id,
  DATE_FORMAT(CAST(created_at AS TIMESTAMP), 'yyyy-MM-dd HH:mm:ss') AS created_at,
  CAST(website_session_id AS INT) AS website_session_id,
  CAST(pageview_url AS STRING) AS pageview_url
FROM data_tables_s.website_pageviews;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Exibindo os valores minimos e maximos de todas as colunas menos a order_item_id
SELECT MIN(created_at) AS Minimo_created_at,
       MAX(created_at) AS Maximo_created_at,
       MIN(website_session_id) AS Minimo_website_session_id,
       MAX(website_session_id) AS Maximo_website_session_id
 FROM data_tables_s.website_pageviews_typed;

Minimo_created_at,Maximo_created_at,Minimo_website_session_id,Maximo_website_session_id
2012-03-19 08:04:16,2015-03-19 07:59:32,1,472871


In [0]:
%sql
-- Contando o total de linhas da tabela
SELECT COUNT(*) FROM data_tables_s.website_pageviews_typed

COUNT(*)
1188124


In [0]:
%sql
-- Realizando a contagem de frequência de cada url
SELECT pageview_url, COUNT(pageview_url) as contagem FROM data_tables_s.website_pageviews_typed GROUP BY pageview_url

pageview_url,contagem
/home,137576
/products,261231
/the-original-mr-fuzzy,162525
/cart,94953
/shipping,64484
/billing,3617
/thank-you-for-your-order,32313
/lander-1,47574
/billing-2,48441
/the-forever-love-bear,26033


## website_sessions

In [0]:
%sql
-- Exibindo os dados da tabela
SELECT * FROM data_tables_s.website_sessions LIMIT 30

website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com
3,2012-03-19 08:26:55,3,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com
4,2012-03-19 08:37:33,4,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com
5,2012-03-19 09:00:55,5,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
6,2012-03-19 09:05:46,6,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com
7,2012-03-19 09:06:27,7,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
8,2012-03-19 09:17:17,8,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
9,2012-03-19 09:27:56,9,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
10,2012-03-19 09:35:37,10,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com


In [0]:
%sql
-- Selecionando todos os dados que possam conter valores NULL para tratá-los.
SELECT * FROM data_tables_s.website_sessions WHERE created_at IS NULL OR is_repeat_session IS NULL OR user_id IS NULL

website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer


In [0]:
%sql
-- Criando a tabela com os dados da tabela order_item com os tipos de dados corretos
CREATE TABLE IF NOT EXISTS data_tables_s.website_sessions_typed AS
SELECT
  CAST(website_session_id AS INT) AS website_session_id,
  DATE_FORMAT(CAST(created_at AS TIMESTAMP), 'yyyy-MM-dd HH:mm:ss') AS created_at,
  CAST(user_id AS INT) AS user_id,
  CAST(is_repeat_session AS INT) AS is_repeat_session
FROM data_tables_s.website_sessions;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Exibindo os primeiros 30 valores da nova tabela
SELECT * FROM data_tables_s.website_sessions_typed LIMIT 30;

website_session_id,created_at,user_id,is_repeat_session
1,2012-03-19 08:04:16,1,0
2,2012-03-19 08:16:49,2,0
3,2012-03-19 08:26:55,3,0
4,2012-03-19 08:37:33,4,0
5,2012-03-19 09:00:55,5,0
6,2012-03-19 09:05:46,6,0
7,2012-03-19 09:06:27,7,0
8,2012-03-19 09:17:17,8,0
9,2012-03-19 09:27:56,9,0
10,2012-03-19 09:35:37,10,0


In [0]:
%sql
-- Exibindo os valores minimos e maximos de todas as colunas menos a order_item_id
SELECT MIN(created_at) AS Minimo_created_at,
       MAX(created_at) AS Maximo_created_at,
       MIN(user_id) AS Minimo_user_id,
       MAX(user_id) AS Maximo_user_id,
       MIN(is_repeat_session) AS Minimo_is_repeat_session,
       MAX(is_repeat_session) AS Maximo_is_repeat_session
 FROM data_tables_s.website_sessions_typed;

Minimo_created_at,Maximo_created_at,Minimo_user_id,Maximo_user_id,Minimo_is_repeat_session,Maximo_is_repeat_session
2012-03-19 08:04:16,2015-03-19 07:59:08,1,394318,0,1


# Criação do esquema estrela

## Criação da tabela fato_order_items

In [0]:
%sql
-- Criação da tabela fato com o esquema dos dados para carga posterior
CREATE TABLE IF NOT EXISTS data_tables_g.fato_order_items (
  order_item_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  order_id BIGINT,
  product_id BIGINT,
  user_id BIGINT,
  website_session_id BIGINT,
  created_at TIMESTAMP,
  price_usd DECIMAL(10,2),
  cogs_usd DECIMAL(10,2)
)

## Criação da dimensão dim_products

In [0]:
%sql

CREATE TABLE IF NOT EXISTS data_tables_g.dim_products(
  product_id BIGINT PRIMARY KEY,
  product_name STRING,
  created_at TIMESTAMP
)

## Criação da dimensão dim_users

In [0]:
%sql

-- Criação da dimensão dim_users apenas com o campo user_id
CREATE TABLE IF NOT EXISTS data_tables_g.dim_users(
  user_id BIGINT PRIMARY KEY
)

## Criação da dimensão dim_date

In [0]:
%sql

-- Criação da dimensão de tempo dim_date
CREATE TABLE IF NOT EXISTS data_tables_g.dim_date(
  date_id DATE PRIMARY KEY,
  year INT,
  month INT,
  day INT
)

## Criação da dimensão dim_pageview

In [0]:
%sql

-- Criando a dimensão dim_pageview
CREATE TABLE IF NOT EXISTS data_tables_g.dim_pageview(
  website_pageview_id BIGINT PRIMARY KEY,
  website_session_id BIGINT,
  user_id BIGINT,
  created_at TIMESTAMP,
  pageview_url VARCHAR(100)
)

## Criação da dimensão dim_refunds

In [0]:
%sql

-- Criando a dimensão dim_refunds
CREATE TABLE IF NOT EXISTS data_tables_g.dim_refunds(
  order_refund_id BIGINT PRIMARY KEY,
  order_id BIGINT,
  user_id BIGINT,
  product_id BIGINT,
  created_at TIMESTAMP
)

## Carga para a tabela fato e dimensões

### Tabela fato_order_items

In [0]:
%sql

-- Preenchendo a tabela fato com as devidas informações
INSERT INTO data_tables_g.fato_order_items (
  order_id,
  product_id,
  user_id,
  website_session_id,  -- Estanciando as colunas para preenchimento
  created_at,
  price_usd,
  cogs_usd
)
SELECT
  oi.order_id,
  oi.product_id,
  o.user_id,
  o.website_session_id,  -- Pegando as informações de duas tabelas, order_items e orders
  o.created_at,
  oi.price_usd,
  oi.cogs_usd
FROM data_tables_s.order_items_typed oi
JOIN data_tables_s.orders_typed o ON oi.order_id = o.order_id  -- Relação entre as tabelas

num_affected_rows,num_inserted_rows
40025,40025


In [0]:
%sql

-- Selecionando os valores das colunas da tabela fato
SELECT * FROM data_tables_g.fato_order_items LIMIT 30

order_item_id,order_id,product_id,user_id,website_session_id,created_at,price_usd,cogs_usd
1,1,1,20,20,2012-03-19T10:42:46.000Z,49.99,19.49
2,2,1,104,104,2012-03-19T19:27:37.000Z,49.99,19.49
3,3,1,147,147,2012-03-20T06:44:45.000Z,49.99,19.49
4,4,1,160,160,2012-03-20T09:41:45.000Z,49.99,19.49
5,5,1,177,177,2012-03-20T11:28:15.000Z,49.99,19.49
6,6,1,232,232,2012-03-20T16:12:47.000Z,49.99,19.49
7,7,1,241,241,2012-03-20T17:03:41.000Z,49.99,19.49
8,8,1,295,295,2012-03-20T23:35:27.000Z,49.99,19.49
9,9,1,304,304,2012-03-21T02:35:01.000Z,49.99,19.49
10,10,1,317,317,2012-03-21T06:45:58.000Z,49.99,19.49


### dim_products

In [0]:
%sql

--Preechendo a dimensão dim_products
INSERT INTO data_tables_g.dim_products (
  product_id,
  product_name,
  created_at
)
SELECT
  product_id,
  product_name,
  created_at
FROM data_tables_s.products_typed

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql

SELECT * FROM data_tables_g.dim_products

product_id,product_name,created_at
1,The Original Mr. Fuzzy,2012-03-19T08:00:00.000Z
2,The Forever Love Bear,2013-01-06T13:00:00.000Z
3,The Birthday Sugar Panda,2013-12-12T09:00:00.000Z
4,The Hudson River Mini bear,2014-02-05T10:00:00.000Z


### dim_refunds

In [0]:
%sql

-- Preenchimento da dimensão dim_refunds

INSERT INTO data_tables_g.dim_refunds (
  order_refund_id,
  order_id,
  user_id,
  product_id,
  created_at
)
SELECT
  r.order_item_refund_id,
  r.order_id,
  o.user_id,
  i.product_id,
  r.created_at
FROM data_tables_s.order_item_refunds_typed r
JOIN data_tables_s.orders_typed o ON r.order_id = o.order_id
JOIN data_tables_s.order_items_typed i ON r.order_id = i.order_id

num_affected_rows,num_inserted_rows
2302,2302


In [0]:
%sql

-- Selecionando os primeiros 30 valores da dimensão dim_refunds
SELECT * FROM data_tables_g.dim_refunds LIMIT 30

order_refund_id,order_id,user_id,product_id,created_at
1,57,1794,1,2012-04-06T11:32:43.000Z
3,71,2247,1,2012-04-15T07:03:48.000Z
2,74,2309,1,2012-04-13T01:09:43.000Z
5,116,3895,1,2012-04-22T20:53:49.000Z
4,118,4065,1,2012-04-17T20:00:37.000Z
6,147,4832,1,2012-05-04T11:59:07.000Z
9,179,6288,1,2012-05-24T16:00:09.000Z
7,186,6381,1,2012-05-12T02:41:14.000Z
8,191,5115,1,2012-05-16T13:06:01.000Z
10,199,6869,1,2012-05-30T17:20:44.000Z


### dim_users

In [0]:
%sql

-- Preenchendo a dimensão dim_users
INSERT INTO data_tables_g.dim_users (
  user_id
)
SELECT DISTINCT
  user_id
FROM data_tables_s.orders_typed


num_affected_rows,num_inserted_rows
31696,31696


In [0]:
%sql

-- Exibindo os valores da dimensão dim_users
SELECT * FROM data_tables_g.dim_users LIMIT 30


user_id
1021
24956
33160
51052
50097
97308
107414
123369
133816
171945


### dim_pageview

In [0]:
%sql

-- Preenchendo a dimensão dim_pageview
INSERT INTO data_tables_g.dim_pageview (
  website_pageview_id,
  website_session_id,
  user_id,
  created_at,
  pageview_url
)
SELECT
  p.website_pageview_id,
  p.website_session_id,
  s.user_id,
  p.created_at,
  p.pageview_url
FROM data_tables_s.website_pageviews_typed p 
JOIN data_tables_s.website_sessions_typed s ON p.website_session_id = s.website_session_id

num_affected_rows,num_inserted_rows
1188124,1188124


In [0]:
%sql

-- Exibindo os valores da dimensão dim_pageview
SELECT * FROM data_tables_g.dim_pageview LIMIT 30

website_pageview_id,website_session_id,user_id,created_at,pageview_url
1,1,1,2012-03-19T08:04:16.000Z,/home
2,2,2,2012-03-19T08:16:49.000Z,/home
3,3,3,2012-03-19T08:26:55.000Z,/home
4,4,4,2012-03-19T08:37:33.000Z,/home
5,5,5,2012-03-19T09:00:55.000Z,/home
6,6,6,2012-03-19T09:05:46.000Z,/home
7,7,7,2012-03-19T09:06:27.000Z,/home
8,6,6,2012-03-19T09:10:08.000Z,/products
9,6,6,2012-03-19T09:10:52.000Z,/the-original-mr-fuzzy
10,6,6,2012-03-19T09:14:02.000Z,/cart


### dim_date

In [0]:
from pyspark.sql.functions import col, dayofmonth, month, year
# Criando um dataframe spark com o range de datas definidos nas variaveis data_inicial e data_final
data_inicial = '2012-01-01'
data_final = '2016-01-01'

df = spark.sql(f"""SELECT explode(sequence(to_date('{data_inicial}'), to_date('{data_final}'), interval 1 day)) AS date_id""")

# Adicionando colunas referente a dia, mes, ano
df_dim_date = df.withColumn('day', dayofmonth(col('date_id'))).withColumn('month', month(col('date_id'))).withColumn('year', year(col('date_id')))

# Exibindo os primeiros 50 valores das colunas

df_dim_date.show(50)



+----------+---+-----+----+
|   date_id|day|month|year|
+----------+---+-----+----+
|2012-01-01|  1|    1|2012|
|2012-01-02|  2|    1|2012|
|2012-01-03|  3|    1|2012|
|2012-01-04|  4|    1|2012|
|2012-01-05|  5|    1|2012|
|2012-01-06|  6|    1|2012|
|2012-01-07|  7|    1|2012|
|2012-01-08|  8|    1|2012|
|2012-01-09|  9|    1|2012|
|2012-01-10| 10|    1|2012|
|2012-01-11| 11|    1|2012|
|2012-01-12| 12|    1|2012|
|2012-01-13| 13|    1|2012|
|2012-01-14| 14|    1|2012|
|2012-01-15| 15|    1|2012|
|2012-01-16| 16|    1|2012|
|2012-01-17| 17|    1|2012|
|2012-01-18| 18|    1|2012|
|2012-01-19| 19|    1|2012|
|2012-01-20| 20|    1|2012|
|2012-01-21| 21|    1|2012|
|2012-01-22| 22|    1|2012|
|2012-01-23| 23|    1|2012|
|2012-01-24| 24|    1|2012|
|2012-01-25| 25|    1|2012|
|2012-01-26| 26|    1|2012|
|2012-01-27| 27|    1|2012|
|2012-01-28| 28|    1|2012|
|2012-01-29| 29|    1|2012|
|2012-01-30| 30|    1|2012|
|2012-01-31| 31|    1|2012|
|2012-02-01|  1|    2|2012|
|2012-02-02|  2|    

In [0]:
# Adicionando os valores do dataframe a dimensão dim_date

df_dim_date.write.mode("append").saveAsTable("workspace.data_tables_g.dim_date")

In [0]:
%sql

-- Exibindo os valores da dimensão dim_date
SELECT * FROM data_tables_g.dim_date

date_id,year,month,day
2012-01-01,2012,1,1
2012-01-02,2012,1,2
2012-01-03,2012,1,3
2012-01-04,2012,1,4
2012-01-05,2012,1,5
2012-01-06,2012,1,6
2012-01-07,2012,1,7
2012-01-08,2012,1,8
2012-01-09,2012,1,9
2012-01-10,2012,1,10


# Analises

In [0]:
%sql

-- Pergunta 1: Qual usuário mais gastou na loja?

SELECT u.user_id, SUM(f.price_usd) AS total_gasto  -- Selecionando os usuarios da dimensão dim_users e a soma do total de gasto deste usuário presente na tabela fato_order_items
FROM data_tables_g.fato_order_items f
JOIN data_tables_g.dim_users u ON f.user_id = u.user_id  -- Relacionando a tabela fato com a dimensão para obter o total de gasto por usuário
GROUP BY u.user_id  -- Agrupando por usuário
ORDER BY total_gasto DESC  -- Ordenando por total de gasto decrescente
LIMIT 1; -- Limitando a uma linha, no caso a linha do topo da tabela.

user_id,total_gasto
341972,251.94


In [0]:
%sql

-- Pergunta 2: Qual é o produto mais vendido da loja?

SELECT p.product_name, COUNT(*) AS qtd_vendida,f.price_usd AS preco_unitario, SUM(f.price_usd) AS Total_vendido_usd -- Selecionando o nome do produto da dimensão dim_products, a quantidade de vezes que ele foi vendido presente na tabela fato_order_items, o preço unitário do produto e o preço total em USD.
FROM data_tables_g.fato_order_items f
JOIN data_tables_g.dim_products p ON f.product_id = p.product_id -- Relacionando a tabela fato com a dimensão produto para obter o nome do produto
GROUP BY p.product_name, f.price_usd  -- Agrupando por nome do produto e preço unitário
ORDER BY qtd_vendida DESC -- Ordenando por quantidade de vezes que o produto foi vendido decrescente
LIMIT 1;  -- Limitando a uma linha

product_name,qtd_vendida,preco_unitario,Total_vendido_usd
The Original Mr. Fuzzy,24226,49.99,1211057.74


In [0]:
%sql

-- Pergunta 3: Qual foi o mês de maior faturamento da loja?

SELECT d.year, d.month, SUM(f.price_usd) AS faturamento  -- Selecionando o ano, mês e o faturamento com a soma do preço unitário presente na tabela fato_order_items
FROM data_tables_g.fato_order_items f
JOIN data_tables_g.dim_date d ON DATE(f.created_at) = d.date_id  -- Relacionando a tabela fato com a dimensão dim_date para obter o ano e mês
GROUP BY d.year, d.month  -- Agrupando por ano e mês
ORDER BY faturamento DESC  -- Ordenando por faturamento decrescente
LIMIT 10;  -- Limitando a 10 linhas

year,month,faturamento
2014,12,144823.02
2015,1,132211.54
2015,2,129212.94
2014,11,128162.98
2014,10,103905.98
2014,9,92232.49
2014,5,88935.27
2014,8,84716.08
2014,7,83288.55
2014,6,80051.25


In [0]:
%sql

-- Pergunta 4: Qual é o item é o mais reembolsado?

SELECT p.product_name, COUNT(f.product_id) AS qtd_reembolsado -- Selecionando o nome do produto da dimensão dim_products e a quantidade de vezes que ele foi reembolsado presente na tabela fato
FROM data_tables_g.fato_order_items f 
JOIN data_tables_g.dim_products p ON f.product_id = p.product_id -- Relacionando a tabela fato com a dimmensão para puxar o nome do produto
JOIN data_tables_g.dim_refunds r ON f.order_id = r.order_id  -- Relacionando a tabela fato com a dimensão dim_refunds para obter o id do pedido e assim ter certeza de que é uma transação de reembolso
GROUP BY p.product_name  -- Agrupando por nome do produto
ORDER BY qtd_reembolsado DESC  -- Ordenando por quantidade de vezes que o produto foi reembolsado decrescente

product_name,qtd_reembolsado
The Original Mr. Fuzzy,1879
The Birthday Sugar Panda,697
The Hudson River Mini bear,542
The Forever Love Bear,326
